8. I denna uppgift arbetar vi med CIFAR-100 datasetet som gicks igenom i kodexempel 1. 
a) Skapa en CNN-modell för att prediktera datasetet. 

b) Om du justerar hyperparametrar med *KerasTuner*, får du bättre resultat? 

c) Prova använd *transfer learning* för att genomföra prediktioner, får du bättre resultat? 

9. I denna uppgift utgår vi ifrån kodexempel 2 i detta kapitel. 

a) Ta egna bilder som du predikterar med en förtränad modell. 

b) Bygg en applikation (med exempelvis *Streamlit*) som använder en förtränad modell för att prediktera bilder. Hur du designar applikationen och vilken funktionalitet du inkluderar väljer du själv. 

10. Använd dig utav CNN för att skapa en modell som kan prediktera teckenspråk. Datan finns tillgänglig här: 
[https://www.kaggle.com/datasets/datamunge/sign-language-mnist](https://www.kaggle.com/datasets/datamunge/sign-language-mnist)

Skapa CNN-modell

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
import matplotlib.pyplot as plt

# 1. Ladda och förbered data
(train_images, train_labels), (test_images, test_labels) = datasets.cifar100.load_data(label_mode='fine')

# Normalisera pixelvärden till mellan 0 och 1
train_images, test_images = train_images / 255.0, test_images / 255.


e:\AI kod\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.0 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
e:\AI kod\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.0 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
e:\AI kod\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.0 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  

In [2]:
# 2. Bygg CNN-modellen
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(100, activation='softmax') # 100 noder för 100 klasser
])

e:\AI kod\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [3]:
# 3. Kompilera och träna
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(train_images, train_labels, epochs=10, 
                    validation_data=(test_images, test_labels))

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.1122 - loss: 3.8474 - val_accuracy: 0.1961 - val_loss: 3.3549
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.2312 - loss: 3.1524 - val_accuracy: 0.2564 - val_loss: 3.0335
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.2975 - loss: 2.8158 - val_accuracy: 0.3078 - val_loss: 2.7804
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.3408 - loss: 2.5996 - val_accuracy: 0.3300 - val_loss: 2.6988
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.3759 - loss: 2.4308 - val_accuracy: 0.3465 - val_loss: 2.6265
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.4051 - loss: 2.2882 - val_accuracy: 0.3417 - val_loss: 2.6580
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.4294 - loss: 2.1650 - val_accuracy: 0.3603 - val_loss: 2.5860
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.4519 - loss: 2.0647 

In [4]:
import keras_tuner as kt

def build_model(hp):
    model = models.Sequential()
    
    # Låter tunern bestämma antalet filter i första lagret alltså mellan 32 och 128. 
    # tunern måste hitta en balans mellan filterna.
    model.add(layers.Conv2D(
        hp.Int('conv_1_filter', min_value=32, max_value=128, step=16),
        (3, 3), activation='relu', input_shape=(32, 32, 3)
    ))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    
    # Låter tunern bestämma antalet noder i Dense-lagret
    model.add(layers.Dense(
        hp.Int('dense_units', min_value=32, max_value=128, step=16), 
        activation='relu'
    ))
    model.add(layers.Dense(100, activation='softmax'))
    
    # Låter tunern testa olika learning rates
    model.compile(optimizer=tf.keras.optimizers.Adam(
        hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])
    
    return model

tuner = kt.Hyperband(build_model,
                     objective='val_accuracy',
                     max_epochs=10,
                     directory='my_dir',
                     project_name='cifar100_tuner')

tuner.search(train_images, train_labels, epochs=5, validation_split=0.2)

# Hämta bästa modellen
best_model = tuner.get_best_models(num_models=1)[0]

Reloading Tuner from my_dir\cifar100_tuner\tuner0.json



e:\AI kod\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [5]:

from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras import layers
import tensorflow as tf

# 1. Definiera Input
# Vi utgår från 32x32 precis som i datan
inputs = tf.keras.Input(shape=(32, 32, 3))

# 2. Förbättringar 
# Skala upp bilden. ResNet behöver ha större bilder.
# Vi gör dem 3 gånger större (96x96).
x = layers.Resizing(96, 96)(inputs)

# Anpassa pixelvärden. 
# min data är 0 till 1 men resnetV2 vill ha -1 till 1 så vi skalar med 2 och flyttar ner med 1
x = layers.Rescaling(scale=2.0, offset=-1.0)(x)

# 3. hämtat resnet basmodell som ska vara bra för detta datasetet
# include_top=False tar bort klassificeraren så vi kan lägga på vår egen
base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=(96, 96, 3))
base_model.trainable = False  # Frys basen

# Kör genom ResNet
x = base_model(x, training=False)

# 4. Bygg klassificeraren 
x = layers.GlobalAveragePooling2D()(x) # tydligen så funkar detta bättre än Flatten här
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.2)(x) # för att minska överanpassning
outputs = layers.Dense(100, activation='softmax')(x)

# 5. Skapa modellen
model_tl = tf.keras.Model(inputs, outputs)

# 6. Kompilera
model_tl.compile(optimizer='adam', 
                 loss='sparse_categorical_crossentropy', 
                 metrics=['accuracy'])

# 7. Träna
print("Startar träning med ResNet50V2.")
history_tl = model_tl.fit(train_images, train_labels, 
                          epochs=10, 
                          validation_data=(test_images, test_labels))

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Startar träning med ResNet50V2 (detta kan ta några minuter)...
Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 164s 102ms/step - accuracy: 0.4179 - loss: 2.3146 - val_accuracy: 0.5015 - val_loss: 1.8634
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 159s 102ms/step - accuracy: 0.5320 - loss: 1.7096 - val_accuracy: 0.5233 - val_loss: 1.7905
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 168s 108ms/step - accuracy: 0.5807 - loss: 1.4875 - val_accuracy: 0.5276 - val_loss: 1.8103
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 162s 104ms/step - accuracy: 0.6189 - loss: 1.3166 - val_accuracy: 0.5232 - val_loss: 1.8582
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 164s 105ms/step - accuracy: 0.6540 - loss: 1.1767 - val_accuracy: 0.5283 - val_loss: 1.9011
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 173s 110ms/step - accuracy: 0.6856 - loss: 1.0483 - val_accuracy: 0.5271 - val_loss: 1.9712
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 164s 105ms/step - accuracy: 0.7118 - los

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np

# get the pre-trained model
# this will know 1000 different common objects
model = MobileNetV2(weights='imagenet')

# Load your image
img_path = 'cat110.jpg'  
# MobileNet wants 224x224 images
img = image.load_img(img_path, target_size=(224, 224))

# 3. Förbehandla bilden
x = image.img_to_array(img)       # convert to numbers
x = np.expand_dims(x, axis=0)     # add another dimension (batch size)
x = preprocess_input(x)           # balance the pixels (-1 to 1)

# 4. Predict
preds = model.predict(x)

# 5. Make it readable 
print('Resultat:', decode_predictions(preds, top=3)[0])

e:\AI kod\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.0 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
e:\AI kod\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.0 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
e:\AI kod\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.0 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
Resultat: [('n02124075', 'Egyptian_cat', np.float32(0.61411905)), ('n02123597', 'Siamese_cat', np.float32(0.21707897)), ('n02127052', 'lynx', np.float32(0.066567324))]


: 